# Capítulo 9 — Frontera eficiente (opcional)

**Objetivo:** comparar pesos actuales con máx Sharpe y mín volatilidad.

### Restricciones
- Long-only, sin apalancamiento, peso ≤ 50%

### Método
Monte Carlo de pesos; Sharpe ≈ μ_p/σ_p (histórico, no predicción).


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
import portfolio_utils as pu
daily = pd.read_csv(ROOT/"data"/"precios_ajustados_diarios.csv", index_col=0, parse_dates=True)
monthly = pu.to_month_end(daily)
rets = monthly.pct_change().loc["2016-01-31":"2026-09-30"]
tickers = list(pu.CURRENT_WEIGHTS)
fr, special = pu.efficient_frontier(rets[tickers].dropna(), n_portfolios=8000, max_weight=0.5)
mu = rets[tickers].mean().values*12
cov = rets[tickers].cov().values*12
w_cur = np.array([pu.CURRENT_WEIGHTS[t] for t in tickers])
ret_c, vol_c = float(w_cur@mu), float(np.sqrt(w_cur@cov@w_cur))
fig, ax = plt.subplots(figsize=(9,6))
sc = ax.scatter(fr.vol*100, fr.ret*100, c=fr.sharpe, cmap="viridis", s=8, alpha=0.35)
plt.colorbar(sc, label="Sharpe")
ax.scatter([vol_c*100],[ret_c*100], c="red", s=140, marker="*", label="Actual", zorder=5)
ax.scatter([special["max_sharpe_stats"]["vol"]*100],[special["max_sharpe_stats"]["ret"]*100], c="orange", s=100, marker="D", label="Máx Sharpe", zorder=5)
ax.scatter([special["min_vol_stats"]["vol"]*100],[special["min_vol_stats"]["ret"]*100], c="cyan", s=100, marker="s", label="Mín vol", zorder=5)
ax.set_xlabel("Vol %"); ax.set_ylabel("Ret %")
ax.set_title("Frontera simulada (long-only, w≤50%)")
ax.legend(); plt.tight_layout()
plt.savefig(ROOT/"figures"/"07_frontera_eficiente.png", dpi=120)
plt.show()
print("Pesos actuales:", {t: round(pu.CURRENT_WEIGHTS[t],3) for t in tickers})
print("Máx Sharpe:", {t: round(float(w),3) for t,w in zip(tickers, special["max_sharpe"])})
print("Mín vol   :", {t: round(float(w),3) for t,w in zip(tickers, special["min_vol"])})


![figura](../figures/07_frontera_eficiente.svg)


Pesos actuales: {'SPYG': 0.305, 'SMH': 0.218, 'BRK-B': 0.204, 'IEMG': 0.202, 'VTI': 0.072}
Máx Sharpe: {'SPYG': 0.035, 'SMH': 0.459, 'BRK-B': 0.445, 'IEMG': 0.047, 'VTI': 0.013}
Mín vol   : {'SPYG': 0.252, 'SMH': 0.001, 'BRK-B': 0.355, 'IEMG': 0.349, 'VTI': 0.043}


**Interpretación:** el óptimo *ex-post* no es invertible como estrategia futura.

## Conclusiones
- Herramienta pedagógica, no botón de maximizar riqueza.
- Tope 50% evita 100% SMH.

## Ejercicios
1. `max_weight=0.35`.
2. Optimiza hasta 2021 y evalúa 2022–2026.
3. Piso IEMG ≥ 10%.
